# P9 serving noise floor\nGreedy decoding (temperature 0). Per task: 3 reruns sent together (batched) and 1 run alone on an idle server (bs1). Any difference is serving noise. Predictions: `quick_runs/predictions/P9.md`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
recs = load_records("P9"); groups = by_task(recs)
print({t: sorted(r["run"] for r in rs) for t, rs in groups.items()})
tt = task_table(groups); tt

{'blade_crofoot': ['batched_a', 'batched_b', 'batched_c', 'bs1'], 'sf_tf01_goal_imbalance': ['batched_a', 'batched_b', 'batched_c', 'bs1'], 'sf_tf03_correlation_causation': ['batched_a', 'batched_b', 'batched_c', 'bs1']}


,task,risk,n,answer_consistency,score_mean,score_sd,first_divergence,consistent_failures,framing.target,framing.population,framing.unit,variables,transformation,model.family,model.controls,validation,metric,verdict,final_selection
0,blade_crofoot,high,4,0.166667,9.166667e-01,0.166667,variables,,1.0,1.0,1.0,0.166667,0.0,1.0,1.0,1.0,1.000000,0.5,0.000000
1,sf_tf01_goal_imbalance,high,4,0.500000,7.878978e-01,0.007359,variables,"checks_class_distribution,uses_class_weight,uses_resampling,uses_stratified_...",1.0,1.0,1.0,0.500000,0.5,1.0,NaN,1.0,0.166667,NaN,0.333333
2,sf_tf03_correlation_causation,low,4,1.000000,2.769464e+07,0.000000,variables,"applies_causal_reasoning,column_x_included",1.0,1.0,1.0,0.333333,1.0,1.0,NaN,1.0,1.000000,NaN,1.000000


In [4]:
# Where do the greedy runs first differ? Compare the model's text step by step against batched_a.
def contents(task, run):
    return [json.loads(l)["content"] for l in open(RUNS / "P9" / task / run / "events.jsonl") if json.loads(l)["event"] == "llm_call"]
div = []
for t in groups:
    ref = contents(t, "batched_a")
    for run in ["batched_b", "batched_c", "bs1"]:
        other = contents(t, run)
        step, char = None, None
        for i, (a, b) in enumerate(zip(ref, other)):
            if a != b:
                step = i; char = next((k for k, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b))); break
        if step is None and len(ref) != len(other): step, char = min(len(ref), len(other)), 0
        div.append({"task": t, "vs_batched_a": run, "identical": step is None, "first_diff_step": step, "first_diff_char": char,
                    "steps_a": len(ref), "steps_other": len(other)})
div = pd.DataFrame(div); div

,task,vs_batched_a,identical,first_diff_step,first_diff_char,steps_a,steps_other
0,blade_crofoot,batched_b,False,2.0,104.0,15,10
1,blade_crofoot,batched_c,False,3.0,0.0,15,8
2,blade_crofoot,bs1,False,0.0,149.0,15,10
3,sf_tf01_goal_imbalance,batched_b,False,0.0,280.0,4,4
4,sf_tf01_goal_imbalance,batched_c,False,0.0,357.0,4,4
5,sf_tf01_goal_imbalance,bs1,False,0.0,219.0,4,4
6,sf_tf03_correlation_causation,batched_b,False,0.0,540.0,2,2
7,sf_tf03_correlation_causation,batched_c,True,NaN,NaN,2,2
8,sf_tf03_correlation_causation,bs1,False,0.0,90.0,2,2


In [5]:
# Answers per run
pd.DataFrame([{"task": r["task_id"], "run": r["run"], "answer": r["answer"], "score": r["score"],
               "checks": r["checks"]} for r in sorted(recs, key=lambda r: (r["task_id"], r["run"]))])

,task,run,answer,score,checks
0,blade_crofoot,batched_a,"(positive, True)",6.666667e-01,"{'dv_matches_expert': True, 'iv_matches_expert': False, 'family_matches_expe..."
1,blade_crofoot,batched_b,"(mixed, True)",1.000000e+00,"{'dv_matches_expert': True, 'iv_matches_expert': True, 'family_matches_exper..."
2,blade_crofoot,batched_c,"(none, False)",1.000000e+00,"{'dv_matches_expert': True, 'iv_matches_expert': True, 'family_matches_exper..."
3,blade_crofoot,bs1,"(mixed, True)",1.000000e+00,"{'dv_matches_expert': True, 'iv_matches_expert': True, 'family_matches_exper..."
4,sf_tf01_goal_imbalance,batched_a,0.792,7.915773e-01,"{'uses_macro_average': True, 'checks_class_distribution': False, 'uses_resam..."
5,sf_tf01_goal_imbalance,batched_b,0.777,7.768595e-01,"{'uses_macro_average': True, 'checks_class_distribution': False, 'uses_resam..."
6,sf_tf01_goal_imbalance,batched_c,0.792,7.915773e-01,"{'uses_macro_average': True, 'checks_class_distribution': False, 'uses_resam..."
7,sf_tf01_goal_imbalance,bs1,0.792,7.915773e-01,"{'uses_macro_average': True, 'checks_class_distribution': False, 'uses_resam..."
8,sf_tf03_correlation_causation,batched_a,27700000.0,2.769464e+07,"{'column_x_included': False, 'applies_causal_reasoning': False}"
9,sf_tf03_correlation_causation,batched_b,27700000.0,2.769464e+07,"{'column_x_included': False, 'applies_causal_reasoning': False}"


In [6]:
summary = {"runs": len(recs), "tasks": list(groups),
  "identical_to_batched_a": int(div.identical.sum()), "compared_pairs": len(div),
  "first_diff": div.to_dict("records"),
  "per_task": tt.set_index("task").to_dict("index"),
  "mean_agreement_by_type": tt[DTYPES].mean().round(3).to_dict(),
  "mean_answer_consistency": round(tt.answer_consistency.mean(), 3)}
json.dump(summary, open(REPO / "quick_runs/results/P9/summary.json", "w"), indent=1, default=str)
summary["identical_to_batched_a"], summary["mean_answer_consistency"], summary["mean_agreement_by_type"]

(1,
 np.float64(0.556),
 {'framing.target': 1.0,
  'framing.population': 1.0,
  'framing.unit': 1.0,
  'variables': 0.333,
  'transformation': 0.5,
  'model.family': 1.0,
  'model.controls': 1.0,
  'validation': 1.0,
  'metric': 0.722,
  'verdict': 0.5,
  'final_selection': 0.444})